# Exact (Hill) cat on the grid of the cat-start map, up to the cat fold (22/10/2026)

Companion of `Hill_satellite_seed_grid.ipynb`, for `lyapunov_heatmap_cat_start.ipynb`. The cat-start map starts from $a(0) = i|\alpha|$, $b(0) = 0$, which is not on the cat orbit,
and its chaos onset lies 6–9% below the cat fold (`Hill_method_cat_fold.ipynb`), where the basin of the cat shrinks. The question is whether a trajectory started **exactly on the cat**
becomes chaotic only at the fold, where the cat disappears with a saddle orbit.

**What this notebook does.** For every $\varepsilon_d/g_2$ of the map ($2{:}0.1{:}13.5$) the cat orbit is followed by pseudo-arclength from the bottom of the map ($\varepsilon_p = 0.2$) towards higher $g_2$,
up to its **fold**. The state at $\tau = 0$ is stored at every $\varepsilon_p$ of the map ($0.2{:}0.01{:}1.1$) below the fold; above the fold the cat does not exist and the entry is `null`
(the map then falls back on $a(0) = i|\alpha|$, $b(0) = 0$). Steps of the arclength that pass the fold (the parameter $s$ turns back) are rejected and halved, so the branch is never continued onto the saddle.
For $\varepsilon_d/g_2 \lesssim 6.4$ the fold is above the top of the map ($g_2^\ast|\alpha| \approx 219$ MHz) and the cat exists on the whole row.

**Start.** The hard-coded cat of `Hill_method_cat_fold` ($\varepsilon_p = 0.5$, $\varepsilon_d = 10.12\,g_2$, converged with the former $J_1$ definition of $g_2$ and continued in $\varepsilon_d$ to the $\sin$ definition),
continued in $\varepsilon_d/g_2$ at $\varepsilon_p = 0.5$, then down to $\varepsilon_p = 0.2$.

**Both cats.** The orbit shifted by half a period, $\mathbf X(\tau + \pi)$, is the other cat (the drive has period $\pi$; the memory harmonics change sign). `state_plus` is the cat near $a(0) = +i|\alpha|$ (the one of `cat_sign = 1`), `state_minus` the other one.

**Conventions** as in the other Hill notebooks: $g_2 = E_J\sin\varepsilon_p\,\varphi_a^2\varphi_b$, $\kappa_b = \pi/10.4$ (code), $\tau = \omega_a t$, $x^\lambda = \varphi x$, $N_H = 16$.
In Julia (`SystemATS`, $t = 0$): $a = (x_a + i\,dx_a/d\tau)/(\sqrt2\varphi_a)$, $b = (x_b + i\,(dx_b/d\tau)/2)/(\sqrt2\varphi_b)$.

**Output** `analysis/output/chicago_poster_2026/cat_seed_hill_22_10_2026.json`. The whole notebook runs headless in about 5 min.

In [ ]:
import json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root
from scipy.integrate import solve_ivp
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import *

OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/satellites/"

w_a = 25.338776456203686
w_b = 2*w_a
phi_a, phi_b = 0.11, 0.204
E_J = 37.12*2*np.pi*1.5
kappa_b = np.pi/10.4                 # 1/ns, code convention (phi_b'' = ... - 2 kappa_b phi_b')
G0 = E_J*phi_a**2*phi_b              # g2 = s G0, s = sin(eps_p)
n = 2
M = np.diag([1.0, 0.25])
K = np.eye(2)

def MHz(g):
    return g/2/np.pi*1e3

## Harmonic balance (copied from `Hill_satellite_seed_grid.ipynb` / `Hill_method_satellite_fold.ipynb`)
`set_s(s, r)` sets $g_2 = sG_0$ and $\varepsilon_d = r\,g_2$.

In [ ]:
class HB:
    def __init__(self, N_H, kfac=1.0, N=None):
        self.N_H, self.N = N_H, N or 64*N_H
        self.C = np.diag([0.0, kfac*kappa_b/w_b])
        dim = n*(2*N_H + 1)
        self.L = np.zeros((dim, dim))
        self.L[:n, :n] = K
        for k in range(1, N_H + 1):
            A, Bk = K - k**2*M, k*self.C
            s_, c_ = 2*k - 1, 2*k
            self.L[s_*n:(s_+1)*n, s_*n:(s_+1)*n] = A
            self.L[s_*n:(s_+1)*n, c_*n:(c_+1)*n] = -Bk
            self.L[c_*n:(c_+1)*n, s_*n:(s_+1)*n] = Bk
            self.L[c_*n:(c_+1)*n, c_*n:(c_+1)*n] = A
        t = np.linspace(0, 2*np.pi, self.N, endpoint=False)
        k = np.arange(1, N_H + 1)
        Phi = np.empty((self.N, 2*N_H + 1))
        Phi[:, 0] = 1/np.sqrt(2)
        Phi[:, 1::2] = np.sin(np.outer(t, k))
        Phi[:, 2::2] = np.cos(np.outer(t, k))
        self.Gamma = np.kron(Phi, np.eye(n))
        self.Gamma_pinv = (2/self.N)*self.Gamma.T

    def set_s(self, s, r):
        self.B = 2*E_J*s*np.array([phi_a**2/w_a, phi_b**2/w_b])          # (beta_a', beta_b')
        self.b_ext = np.zeros(n*(2*self.N_H + 1))
        self.b_ext[2*2*n + 1] = -np.sqrt(2)*r*s*G0*phi_b/w_b             # c_2 block of mode b

    def X(self, z):
        return (self.Gamma @ z).reshape(self.N, n).T

    def residual(self, z):
        X = self.X(z)
        Fnl = -np.sqrt(2)*self.B[:, None]*(np.cos(np.sqrt(2)*(X[0] + X[1])) - 1)[None, :]
        return self.L @ z + self.Gamma_pinv @ Fnl.T.reshape(-1) - self.b_ext

    def jacobian(self, z):
        X = self.X(z)
        sn = np.sin(np.sqrt(2)*(X[0] + X[1]))
        JG = np.einsum('j,ab,jbm->jam', 2*sn, np.outer(self.B, [1, 1]),
                       self.Gamma.reshape(self.N, n, -1)).reshape(n*self.N, -1)
        return self.L + self.Gamma_pinv @ JG

    def solve(self, z0, tol=1e-12):
        sol = root(self.residual, z0, jac=self.jacobian, method="hybr", tol=tol, options={"maxfev": 100*(len(z0) + 1)})
        if not sol.success:
            raise RuntimeError(sol.message)
        return sol.x


def orbit(z, N_H, t):
    k = np.arange(1, N_H + 1)
    out = []
    for zz in (z[::2], z[1::2]):
        s_, c_ = zz[1::2], zz[2::2]
        x = zz[0]/np.sqrt(2) + np.sin(np.outer(t, k)) @ s_ + np.cos(np.outer(t, k)) @ c_
        v = np.cos(np.outer(t, k)) @ (k*s_) - np.sin(np.outer(t, k)) @ (k*c_)
        out.append((x, v))
    return out

def strobe(z, N_H, tau=0.0):
    (x_a, v_a), (x_b, v_b) = orbit(z, N_H, np.array([tau]))
    return np.array([x_a[0], x_b[0], v_a[0], v_b[0]])

def memory_phase(z):
    za = z[::2]
    return np.angle(za[2] + 1j*za[1])/np.pi

def mu_monodromy(hbk, z):
    # the four Floquet multipliers of the HB orbit z (variational equation over one period)
    Minv, base = np.linalg.inv(M), np.outer(hbk.B, [1, 1])
    def rhs(t, Y):
        (x_a, _), (x_b, _) = orbit(z, hbk.N_H, np.atleast_1d(t))
        A = np.zeros((4, 4)); A[:2, 2:] = np.eye(2)
        A[2:, :2] = -Minv @ (K + 2*np.sin(np.sqrt(2)*(x_a[0] + x_b[0]))*base); A[2:, 2:] = -Minv @ hbk.C
        return (A @ Y.reshape(4, 4)).reshape(-1)
    sol = solve_ivp(rhs, (0, 2*np.pi), np.eye(4).reshape(-1), method="DOP853", rtol=1e-10, atol=1e-12)
    return np.linalg.eigvals(sol.y[:, -1].reshape(4, 4))

def full_rhs(s, r, kfac=1.0):
    beta = 2*E_J*s*np.array([phi_a**2/w_a, phi_b**2/w_b])
    eps_d_p, kp = r*s*G0*phi_b/w_b, kfac*kappa_b/w_b
    def f(t, Y):
        x_a, x_b, v_a, v_b = Y
        g = np.cos(np.sqrt(2)*(x_a + x_b)) - 1
        return [v_a, v_b, -x_a + np.sqrt(2)*beta[0]*g, 4*(-x_b - np.sqrt(2)*eps_d_p*np.cos(2*t) + np.sqrt(2)*beta[1]*g - kp*v_b)]
    return f


def solve_robust(hbk, z):
    # Powell sometimes stalls at tol = 1e-12 when the guess is already converged to ~1e-8
    try:
        return hbk.solve(z)
    except RuntimeError:
        return hbk.solve(z, tol=1e-9)

## Cat seed and continuation up to the fold
`cat_column` is `satellite_column` of `Hill_satellite_seed_grid` run the other way: towards higher $s$, stopping at the first turning point.

In [ ]:
from scipy.special import jv

z_seed_cat = np.array([-2.09501523e-02, -3.60273363e-02,  3.26246220e-01, -3.28386996e-17,
       -6.78344780e-03,  1.43507376e-17, -1.72514305e-04, -3.13979476e-03,
       -4.23320965e-03, -7.46552651e-02, -8.01435361e-04, -8.82665625e-03,
        5.05426053e-05,  4.29993690e-04,  7.79954440e-06,  6.01698525e-05,
        1.00817384e-04,  8.67341068e-04])   # cat at eps_p = 0.5, eps_d = 10.12 g2 with the former J1 g2 (Hill_method_cat_fold)

def cat_at_half(hb, eps_p=0.5, r0=10.12):
    # hard-coded seed, continued in eps_d from the J1 to the sin definition of g2, at fixed eps_p
    s = np.sin(eps_p)
    z = np.pad(z_seed_cat, (0, n*(2*hb.N_H + 1) - len(z_seed_cat)))
    for r in np.linspace(r0*jv(1, eps_p)/s, r0, 25):
        hb.set_s(s, r); z = hb.solve(z)
    return z

def cat_column(hb, z, s, r, s_grid, ds_max=0.01, ds_fold=1e-6):
    """Pseudo-arclength along the cat branch at fixed eps_d/g2 = r, from (z, s) towards higher s, up to the fold.
    Returns the HB coefficients at every s of s_grid (NaN rows above the fold or below s), the s of the fold (NaN if none below s_grid.max()).
    A step after which s decreases has passed the fold: it is rejected and halved, so the branch is never continued onto the saddle."""
    def F(y):
        hb.set_s(y[-1], r); return hb.residual(y[:-1])
    def dF(y):
        hb.set_s(y[-1], r); Jz, r_ = hb.jacobian(y[:-1]), hb.residual(y[:-1])
        h = 1e-7
        hb.set_s(y[-1] + h, r)
        return np.column_stack([Jz, (hb.residual(y[:-1]) - r_)/h])
    def tangent(y, t_prev=None):
        t = np.linalg.svd(dF(y))[2][-1]
        if (t_prev is not None and t @ t_prev < 0) or (t_prev is None and t[-1] < 0):
            t = -t
        return t
    zs = np.full((len(s_grid), len(z)), np.nan)
    zs[np.isclose(s_grid, s, rtol=0, atol=1e-12)] = z
    y = np.append(z, s); t = tangent(y); ds = 0.003
    while True:
        sol = root(lambda w: np.append(F(w), t @ (w - y) - ds), y + ds*t,
                   jac=lambda w: np.vstack([dF(w), t]), method="hybr", tol=1e-12)
        t_new = tangent(sol.x, t) if sol.success else None
        if not sol.success or t_new[-1] < 0 or sol.x[-1] <= y[-1]:   # failed, or passed the fold
            ds /= 2
            if ds < ds_fold:
                return zs, y[-1]
            continue
        for g in np.where((s_grid > y[-1]) & (s_grid <= sol.x[-1]))[0]:   # grid values of s crossed by this step
            w = (s_grid[g] - y[-1])/(sol.x[-1] - y[-1])
            guess = y[:-1] + w*(sol.x[:-1] - y[:-1])
            hb.set_s(s_grid[g], r); zs[g] = solve_robust(hb, guess)
            assert np.linalg.norm(zs[g] - guess) < 1e-2*np.linalg.norm(guess), "grid solve left the branch"
        y, t = sol.x, t_new; ds = min(1.3*ds, ds_max)
        if y[-1] > s_grid.max():
            return zs, np.nan                    # no fold inside the map

## All columns of the map

In [ ]:
eps_p_list = np.round(np.arange(0.2, 1.1 + 1e-9, 0.01), 2)      # = Julia vcat(0.2:.01:1.1)
ratio_list = np.round(np.arange(2.0, 13.5 + 1e-9, 0.1), 1)      # = Julia 2:.1:13.5
s_grid = np.sin(eps_p_list)
i_half = np.argmin(np.abs(eps_p_list - 0.5))

N_H = 16
hb = HB(N_H)
z10 = cat_at_half(hb)                                            # eps_p = 0.5, eps_d = 10.12 g2
z_bottom = {}                                                    # cat at eps_p = 0.2 for every ratio
for part in (ratio_list[ratio_list > 10.12], ratio_list[ratio_list < 10.12][::-1]):
    z, r_prev = z10.copy(), 10.12
    for r in part:
        for rr in np.linspace(r_prev, r, 3)[1:]:
            hb.set_s(s_grid[i_half], rr); z = hb.solve(z)
        r_prev = r
        zb = z.copy()
        for i in range(i_half - 1, -1, -1):
            hb.set_s(s_grid[i], r); zb = hb.solve(zb)
        z_bottom[r] = zb

n_r, n_p = len(ratio_list), len(eps_p_list)
state_plus = np.full((n_r, n_p, 4), np.nan)       # [x_a, x_b, dx_a/dtau, dx_b/dtau] at tau = 0, cat near a(0) = +i|alpha|
state_minus = np.full((n_r, n_p, 4), np.nan)      # the other cat (orbit at tau = pi)
mu_all = np.full((n_r, n_p, 4), np.nan + 0j)
s_fold = np.full(n_r, np.nan)
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # overflows inside rejected Powell iterations
    for j, r in enumerate(ratio_list):
        zs, s_fold[j] = cat_column(hb, z_bottom[r], s_grid[0], r, s_grid)
        for i in np.where(np.isfinite(zs[:, 0]))[0]:
            y0, ypi = strobe(zs[i], N_H, 0.0), strobe(zs[i], N_H, np.pi)
            if ypi[2] > y0[2]:                       # Im a(0) ∝ dx_a/dtau: keep the cat with Im a(0) > 0 as "plus"
                y0, ypi = ypi, y0
            state_plus[j, i], state_minus[j, i] = y0, ypi
            hb.set_s(s_grid[i], r); mu_all[j, i] = mu_monodromy(hb, zs[i])
g2_fold = MHz(s_fold*G0)
max_abs_mu = np.abs(mu_all).max(axis=2)
has = np.isfinite(state_plus[..., 0])
print(f"{time.time() - t0:.0f} s; cat on {has.sum()}/{has.size} grid points ({100*has.mean():.1f}%), stable (max|mu| < 1) on {(max_abs_mu < 1).sum()}")
print(f"cat fold inside the map for eps_d/g2 >= {ratio_list[np.isfinite(g2_fold)].min()}: "
      f"{np.nanmax(g2_fold):.2f} MHz .. {np.nanmin(g2_fold):.2f} MHz (eps_d/g2 = {ratio_list[-1]})")

## Checks
1. The fold against `cat_fold_line_19_10_2026.json` (`Hill_method_cat_fold`, $N_H = 12$, ratios $2{:}0.25{:}13.5$; compared where the two grids coincide).
2. The cat is stable on the whole stored range (max|μ| < 1), up to the last grid point below the fold.
3. The stored state is on the orbit: time integration of the full equations over one period, at the highest grid point of each column.
4. `state_plus` is the cat of `cat_sign = 1`: $a(0)$ close to $+i|\alpha|$, $|\alpha|^2 = \varepsilon_d/2g_2$.

In [ ]:
with open(OUT_DIR + "cat_fold_line_19_10_2026.json") as f:
    ref = json.load(f)
ref_g = {round(r, 2): (np.nan if g is None else g) for r, g in zip(ref["eps_d_over_g2"], ref["g2_fold_MHz"])}
common = [j for j, r in enumerate(ratio_list) if round(r, 2) in ref_g]
d = np.array([g2_fold[j] - ref_g[round(ratio_list[j], 2)] for j in common])
top = MHz(s_grid[-1]*G0)                    # the reference is NaN only without a fold below g2_max = 137 MHz: compare with the top of the map
both_nan = np.array([np.isnan(g2_fold[j]) == (not ref_g[round(ratio_list[j], 2)] <= top) for j in common])
print(f"1. fold vs cat_fold_line_19_10_2026 at {len(common)} common ratios: max |difference| = {np.nanmax(np.abs(d)):.3f} MHz; "
      f"same set of ratios without a fold inside the map: {both_nan.all()}")

print(f"2. max|mu| on the stored cats: {np.nanmin(max_abs_mu):.3f} .. {np.nanmax(max_abs_mu):.4f}")

i_hi = n_p - 1 - np.argmax(has[:, ::-1], axis=1)                 # highest eps_p with a cat, per ratio
err = []
for j, r in enumerate(ratio_list):
    y0 = state_plus[j, i_hi[j]]
    y1 = solve_ivp(full_rhs(s_grid[i_hi[j]], r), (0, 2*np.pi), y0, method="DOP853", rtol=1e-11, atol=1e-13).y[:, -1]
    err.append(np.linalg.norm(y1 - y0)/np.linalg.norm(y0))
print(f"3. one-period return error, relative: max {np.max(err):.1e}")

a0 = (state_plus[..., 0] + 1j*state_plus[..., 2])/(np.sqrt(2)*phi_a)
alpha = np.sqrt(ratio_list/2)[:, None]
print(f"4. a(0)/(i|alpha|): real part {np.nanmin((a0/(1j*alpha)).real):.3f} .. {np.nanmax((a0/(1j*alpha)).real):.3f}, "
      f"arg/pi {np.nanmin(np.angle(a0/(1j*alpha)))/np.pi:+.3f} .. {np.nanmax(np.angle(a0/(1j*alpha)))/np.pi:+.3f}")

In [ ]:
g2_MHz_list = MHz(s_grid*G0)
alpha_sq = ratio_list/2
fig, axes = plt.subplots(1, 2, figsize=(2*figsize[0], figsize[1]))
ax = axes[0]
pc = ax.pcolormesh(g2_MHz_list, alpha_sq, np.abs(a0)/np.sqrt(alpha_sq)[:, None], cmap="viridis", shading="nearest")
fig.colorbar(pc, ax=ax, label=r"$|a(0)|/|\alpha|$")
ax.set_title("Seed: exact cat at $t = 0$", fontsize=8)
ax = axes[1]
pc = ax.pcolormesh(g2_MHz_list, alpha_sq, max_abs_mu, cmap="viridis", shading="nearest")
fig.colorbar(pc, ax=ax, label=r"$\max|\mu|$")
ax.set_title("Floquet stability of the cat", fontsize=8)
for ax in axes:
    ax.plot(g2_fold, alpha_sq, "k--", lw=1, label="cat fold (Hill)")
    ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\varepsilon_d/2g_2$")
    ax.set_xlim(g2_MHz_list.min(), g2_MHz_list.max()); ax.set_ylim(alpha_sq.min(), alpha_sq.max())
axes[0].legend(fontsize=5, loc="lower left")
plt.tight_layout()
plt.savefig(FIG_DIR + "cat_seed_hill_grid_22_10_2026.pdf")
plt.show()

## Export
`null` above the fold. Outer index: $\varepsilon_d/g_2$ (`j`), inner index: $\varepsilon_p$ (`i`), i.e. `state_plus[j][i]`, as `Lyap_coef_chaos[j, i]` in the Julia map. `g2_fold_MHz[j]` is `null` when the fold is above the map.

In [ ]:
nan2none = lambda a: [[None if not np.all(np.isfinite(v)) else (v.tolist() if np.ndim(v) else float(v)) for v in row] for row in a]
with open(OUT_DIR + "cat_seed_hill_22_10_2026.json", "w") as f:
    json.dump({"description": "Exact period-1 cat (harmonic balance, pseudo-arclength from eps_p = 0.2 up to its fold) "
                              "on the grid of lyapunov_heatmap_cat_start.ipynb; Hill_cat_seed_grid.ipynb",
               "g2_definition": "E_J sin(eps_p) phi_a^2 phi_b", "kappa_b": kappa_b, "N_H": N_H,
               "state_definition": "[x_a, x_b, dx_a/dtau, dx_b/dtau] at tau = omega_a t = 0, x^lambda = phi x; "
                                   "a = (x_a + i dx_a/dtau)/(sqrt2 phi_a), b = (x_b + i (dx_b/dtau)/2)/(sqrt2 phi_b); null above the fold. "
                                   "state_plus: cat with Im a(0) > 0 (cat_sign = 1), state_minus: the other cat",
               "eps_p_list": eps_p_list.tolist(), "eps_d_over_g2": ratio_list.tolist(), "g2_MHz": g2_MHz_list.tolist(),
               "state_plus": nan2none(state_plus), "state_minus": nan2none(state_minus), "max_abs_mu": nan2none(max_abs_mu),
               "s_fold": [None if np.isnan(v) else float(v) for v in s_fold],
               "g2_fold_MHz": [None if np.isnan(v) else float(v) for v in g2_fold]}, f)
print("written", OUT_DIR + "cat_seed_hill_22_10_2026.json")